### Import libraries

In [1]:
import json
import pandas as pd

### Load the raw OSM JSON export

In [2]:
with open('../data/raw/export.json') as f:
    data = json.load(f)

print("Total elements found:", len(data['elements']))

Total elements found: 24743


### Extract and clean the relevant fields

In [3]:
rows = []

for el in data["elements"]:
    tags = el.get("tags", {})

    # Only keep actual settlements
    if tags.get("place") not in ["city", "town", "village", "hamlet"]:
        continue

    rows.append({
        "id": el.get("id"),
        "name": tags.get("name:en", tags.get("name")),
        "place_type": tags.get("place"),
        "latitude": el.get("lat"),
        "longitude": el.get("lon"),
        "population": tags.get("population")
    })

df = pd.DataFrame(rows)

print("Settlement records:", len(df))
df.head()

Settlement records: 24743


,id,name,place_type,latitude,longitude,population
0,58876734,Landi Kotal,town,34.100527,71.146850,33697
1,66319295,Ghakhar Mandi,town,32.304166,74.146501,NaN
2,81842063,Adiala,town,33.457523,72.995594,NaN
3,81844596,Khasala Khurd,village,33.439479,72.973001,NaN
4,90529210,Kamra,village,33.856090,72.394136,3917


### Check data quality

In [4]:
print("Missing names:", df["name"].isna().sum())
print("Missing coordinates:",
      df[["latitude", "longitude"]].isna().sum().sum())

print("Duplicate name + coordinates:",
      df.duplicated(
          subset=["name", "latitude", "longitude"]
      ).sum())

Missing names: 953
Missing coordinates: 0
Duplicate name + coordinates: 1


### Remove duplicates

In [5]:
df = df.drop_duplicates(
    subset=["name", "latitude", "longitude"]
).copy()

print("Final settlement rows:", len(df))

Final settlement rows: 24742


### Save the cleaned CSV

In [6]:
df.to_csv(
    "../data/processed/osm_settlements_cleaned.csv",
    index=False
)

## Calculate nearest settlement

In [7]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Load lake dataset
lakes = pd.read_csv('../data/processed/glofguard_lakes_v1.csv')

# Use the already-cleaned settlement dataframe
settlements = df.copy()

print("Lakes:", lakes.shape)
print("Settlements:", settlements.shape)

# Convert latitude/longitude to radians
lake_coords = np.radians(
    lakes[['latitude', 'longitude']].values
)

settlement_coords = np.radians(
    settlements[['latitude', 'longitude']].values
)

# Build BallTree using Haversine distance
tree = BallTree(
    settlement_coords,
    metric='haversine'
)

# Find nearest settlement for every lake
distances, indices = tree.query(
    lake_coords,
    k=1
)

# Convert radians → kilometers
lakes['distance_to_nearest_settlement_km'] = (
    distances.flatten() * 6371
)

# Get name of nearest settlement
lakes['nearest_settlement_name'] = (
    settlements.iloc[indices.flatten()]['name'].values
)

# Check results
print(
    lakes[
        [
            'latitude',
            'longitude',
            'distance_to_nearest_settlement_km',
            'nearest_settlement_name'
        ]
    ].head(10)
)

print("\nDistance summary:")
print(
    lakes['distance_to_nearest_settlement_km'].describe()
)

Lakes: (8806, 9)
Settlements: (24742, 6)
    latitude  longitude  distance_to_nearest_settlement_km  \
0  34.828994  74.061891                           9.746248   
1  34.820079  74.086340                           7.396566   
2  34.806459  74.071990                           7.619067   
3  34.857497  74.076763                           8.773070   
4  34.861163  74.078262                           8.521195   
5  34.862122  74.077092                           8.595121   
6  34.856478  74.087372                           7.901622   
7  34.912142  74.089000                           7.924807   
8  34.909833  74.089830                           7.751837   
9  34.939855  74.073015                          10.759006   

  nearest_settlement_name  
0            Khawaja Seri  
1            Khawaja Seri  
2            Khawaja Seri  
3                 Bakwali  
4                 Bakwali  
5                 Bakwali  
6                 Bakwali  
7                 Bakwali  
8                 Bakwal

In [8]:
print("\nNumber of lakes:", len(lakes))

print(
    "Missing distances:",
    lakes['distance_to_nearest_settlement_km'].isna().sum()
)

print(
    "Missing settlement names:",
    lakes['nearest_settlement_name'].isna().sum()
)


Number of lakes: 8806
Missing distances: 0
Missing settlement names: 817


### Save The File

In [9]:
lakes.to_csv('../data/processed/glofguard_lakes_v2_settlement.csv', index=False)
print("Saved: glofguard_lakes_v2_settlement.csv")

Saved: glofguard_lakes_v2_settlement.csv
